In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.ensemble import RandomForestRegressor

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# Load weekly SKU dataset
weekly_df = pd.read_csv('../data/processed/weekly_sku_demand.csv')
weekly_df['Week_Start'] = pd.to_datetime(weekly_df['Week_Start'])
weekly_df = weekly_df.sort_values(['StockCode', 'Week_Start']).reset_index(drop=True)

# Target same SKU as Phase 7 for direct model comparison
top_sku = weekly_df.groupby('StockCode')['Quantity'].sum().idxmax()
sku_data = weekly_df[weekly_df['StockCode'] == top_sku].copy().reset_index(drop=True)

print(f"Loaded modeling dataset for SKU: {top_sku} ({len(sku_data)} total weekly observations).")

Loaded modeling dataset for SKU: 84077 (103 total weekly observations).


In [2]:
# 1. Weekly Lag Features
# In weekly data: lag_1 = 1 week ago, lag_2 = 2 weeks ago, lag_4 = 4 weeks ago
sku_data['lag_1'] = sku_data['Quantity'].shift(1)
sku_data['lag_2'] = sku_data['Quantity'].shift(2)
sku_data['lag_4'] = sku_data['Quantity'].shift(4)

# 2. Rolling Aggregations (Computed ONLY over lagged values to strictly prevent leakage)
sku_data['rolling_mean_4'] = sku_data['lag_1'].rolling(window=4).mean()
sku_data['rolling_std_4']  = sku_data['lag_1'].rolling(window=4).std()

# 3. Calendar Features
sku_data['month'] = sku_data['Week_Start'].dt.month
sku_data['quarter'] = sku_data['Week_Start'].dt.quarter
sku_data['week_of_year'] = sku_data['Week_Start'].dt.isocalendar().week.astype(int)

# Drop rows with NaNs caused by lag/rolling shifts
model_df = sku_data.dropna().reset_index(drop=True)

print(f"Feature engineering complete. Total rows available for ML modeling: {len(model_df)}")
model_df[['Week_Start', 'Quantity', 'lag_1', 'lag_2', 'rolling_mean_4', 'month']].head()

Feature engineering complete. Total rows available for ML modeling: 99


,Week_Start,Quantity,lag_1,lag_2,rolling_mean_4,month
0,2010-01-18,384,336.00,480.00,711.75,1
1,2010-01-25,627,384.00,336.00,601.00,1
2,2010-02-01,240,627.00,384.00,456.75,2
3,2010-02-08,480,240.00,627.00,396.75,2
4,2010-02-15,288,480.00,240.00,432.75,2


In [3]:
# Train/Validation Split (Matching 80% / 20% chronological logic from Phase 7)
split_idx = int(len(model_df) * 0.80)

train_df = model_df.iloc[:split_idx].copy()
val_df   = model_df.iloc[split_idx:].copy()

# Define feature sets
feature_cols = ['lag_1', 'lag_2', 'lag_4', 'rolling_mean_4', 'rolling_std_4', 'month', 'quarter', 'week_of_year']

X_train, y_train = train_df[feature_cols], train_df['Quantity']
X_val, y_val     = val_df[feature_cols], val_df['Quantity']

print(f"ML Training Range:   {train_df['Week_Start'].min().date()} to {train_df['Week_Start'].max().date()} ({len(X_train)} weeks)")
print(f"ML Validation Range: {val_df['Week_Start'].min().date()} to {val_df['Week_Start'].max().date()} ({len(X_val)} weeks)")

ML Training Range:   2010-01-18 to 2011-07-25 (79 weeks)
ML Validation Range: 2011-08-01 to 2011-12-12 (20 weeks)


In [5]:
# Fit Holt-Winters Exponential Smoothing (Additive Trend + Additive Seasonality with 52-week annual cycle)
# Using simple additive trend for stable small-sample convergence
hw_model = ExponentialSmoothing(
    train_df['Quantity'],
    trend='add',
    seasonal=None,  # Seasonal set to None due to train sample length < 104 weeks
    initialization_method='estimated'
).fit()

# Predict over validation set length
val_df['Forecast_HoltWinters'] = hw_model.forecast(steps=len(val_df)).values

In [6]:
# Train Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=5)
rf_model.fit(X_train, y_train)

# Predict on validation features
val_df['Forecast_RandomForest'] = rf_model.predict(X_val)

# Feature Importance Inspection
importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("--- Random Forest Feature Importance ---")
display(importance_df)

--- Random Forest Feature Importance ---


,Feature,Importance
2,lag_4,0.44
0,lag_1,0.11
4,rolling_std_4,0.11
3,rolling_mean_4,0.10
1,lag_2,0.07
7,week_of_year,0.07
5,month,0.06
6,quarter,0.03


In [7]:
def calculate_metrics(y_true, y_pred, model_name):
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    wape = (np.sum(np.abs(y_true - y_pred)) / np.sum(y_true)) * 100
    return {
        'Model': model_name,
        'MAE (Units)': round(mae, 2),
        'RMSE (Units)': round(rmse, 2),
        'WAPE (%)': round(wape, 2)
    }

# Load Phase 7 Baseline Metrics for comparison
baseline_df = pd.read_csv('../reports/results/baseline_performance.csv')

# Calculate metrics for advanced models
advanced_metrics = [
    calculate_metrics(val_df['Quantity'], val_df['Forecast_HoltWinters'], 'Holt-Winters Exp Smoothing'),
    calculate_metrics(val_df['Quantity'], val_df['Forecast_RandomForest'], 'Random Forest Regressor')
]

# Combine all results into a single benchmark summary table
all_results = pd.concat([baseline_df, pd.DataFrame(advanced_metrics)], ignore_index=True)
all_results = all_results.sort_values('WAPE (%)').reset_index(drop=True)

print("=== COMPLETE MODEL BENCHMARK TABLE (VALIDATION SET) ===")
display(all_results)

=== COMPLETE MODEL BENCHMARK TABLE (VALIDATION SET) ===


,Model,MAE (Units),RMSE (Units),WAPE (%)
0,4-Week Moving Average,325.50,355.09,50.34
1,Holt-Winters Exp Smoothing,566.03,1080.44,55.85
2,Naive Forecast,489.40,543.46,75.69
3,Random Forest Regressor,775.92,1263.85,76.55
4,Seasonal Naive,765.90,1137.43,118.45


In [ ]:
# Save updated benchmark results
results_dir = '../reports/results'
if os.path.exists(results_dir) and not os.path.isdir(results_dir):
    os.remove(results_dir)
os.makedirs(results_dir, exist_ok=True)

all_results.to_csv(os.path.join(results_dir, 'model_performance_summary.csv'), index=False)

# Visualization
plt.figure(figsize=(12, 5))
plt.plot(val_df['Week_Start'], val_df['Quantity'], label='Actual Demand', color='black', linewidth=2)
plt.plot(val_df['Week_Start'], val_df['Forecast_HoltWinters'], label='Holt-Winters', linestyle='--', color='blue')
plt.plot(val_df['Week_Start'], val_df['Forecast_RandomForest'], label='Random Forest', linestyle='--', color='green')

plt.title(f'Advanced Demand Forecast vs Actuals (SKU: {top_sku})', fontsize=12, fontweight='bold')
plt.xlabel('Week Start Date')
plt.ylabel('Weekly Units Demanded')
plt.legend()
plt.tight_layout()
plt.show()

print("Model performance summary saved to reports/results/model_performance_summary.csv.")